# Query a Fabric Ontology through MCP

Connect directly to the Fabric IQ ontology MCP endpoint, inspect the tools it exposes, and call one of those tools with a natural-language question. The endpoint uses the signed-in Azure Developer CLI identity and requires access to the Fabric workspace.

## Step 1: Load configuration and create an MCP session helper

Load the Fabric tenant and ontology MCP endpoint written to the project `.env` file during provisioning. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

In [15]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_ONTOLOGY_MCP_URL = os.environ["FABRIC_ONTOLOGY_MCP_URL"]
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"


@asynccontextmanager
async def open_ontology_session():
    credential = AzureDeveloperCliCredential(tenant_id=FABRIC_TENANT_ID)
    token = credential.get_token(FABRIC_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(FABRIC_ONTOLOGY_MCP_URL, http_client=http_client,) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Fabric ontology configuration loaded")

Fabric ontology configuration loaded


## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the Fabric ontology, including each tool's input schema.

In [16]:
import json

from rich.console import Console

console = Console()

async with open_ontology_session() as session:
    tools_response = await session.list_tools()

ontology_tools = tools_response.tools

for tool in ontology_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

Tool: list_ontology_entity_types

Description: Retrieves the specified or all entity types in the ontology and their details—such as property names, 
property types, and available telemetry. Helps users explore the schema of the ontology

Input schema:

{
  "type": "object",
  "properties": {
    "entityName": {
      "type": "string",
      "description": "Optional value of the exact name of the entity to retrieve, as it appears in the ontology. Must start with a letter and contain only letters, digits, hyphens, or underscores.",
      "minLength": 1,
      "maxLength": 128,
      "pattern": "^[a-zA-Z][a-zA-Z0-9_-]{0,127}$"
    },
    "includeProperties": {
      "type": "boolean",
      "description": "If true, it includes the properties of the entities"
    }
  },
  "required": []
}

Tool: search_ontology

Description: Retrieves the answer to a natural language question from the ontology data estate, returns raw results
in JSON format and optionally a derived natural language answer

Input schema:

{
  "type": "object",
  "properties": {
    "naturalLanguageQuery": {
      "type": "string",
      "description": "Natural language question to translate to Ontology query language and execute against the Ontology",
      "minLength": 3,
      "maxLength": 10000
    },
    "naturalLanguageResponse": {
      "type": "boolean",
      "description": "Optional flag; if true, returns a natural language summary of the query results"
    }
  },
  "required": [
    "naturalLanguageQuery"
  ]
}

## Step 3: Query the Fabric ontology

Call the ontology's `search_ontology` tool with a natural-language question and request a summarized response. Change `question` to explore other product and inventory questions.

In [17]:
question = "Which products currently have the lowest inventory quantities?"

async with open_ontology_session() as session:
    result = await session.call_tool(
        "search_ontology",
        {
            "naturalLanguageQuery": question,
            "naturalLanguageResponse": True,
        },
    )

response = [block.text for block in result.content if block.type == "text"]
console.print_json(data=json.loads(response[0]))

{
  "raw": {
    "Fields": [
      "Store_json",
      "Store_id",
      "Inventory_json",
      "Inventory_id",
      "Product_json",
      "Product_id",
      "available_quantity"
    ],
    "Value": [
      [
        "{\"labels\":[\"Store\"],\"oid\":\"13510798882111496\",\"properties\":{\"city\":\"Spokane\",\"customerDistributionWeight\":8.0,\"orderFrequencyMultiplier\":2.0,\"orderValueMultiplier\":1.0,\"region\":\"Eastern Washington\",\"rlsUserId\":\"d8e9f0a1-b2c3-4567-8901-234567890abc\",\"state\":\"WA\",\"storeId\":\"STORE-SPO\",\"storeName\":\"Zava Retail Spokane\",\"storeType\":\"Retail\"}}",
        "STORE-SPO",
        "{\"labels\":[\"Inventory\"],\"oid\":\"56013520365420551\",\"properties\":{\"availableQuantity\":0,\"inventoryId\":\"INV-STORE-SPO-HWHL000036\",\"lastRestockedAt\":\"2026-06-10T00:00:00.000+00:00\",\"quantityOnHand\":1,\"quantityReserved\":1,\"reorderPoint\":24,\"reorderQuantity\":48,\"sku\":\"HWHL000036\",\"storeId\":\"STORE-SPO\"}}",
        "INV-STORE-SPO-HWHL000036",
        "{\"labels\":[\"Product\"],\"oid\":\"1125899906842625\",\"properties\":{\"category\":\"HARDWARE\",\"description\":\"Heavy-duty hasp and staple set for padlock security on gates, sheds, and storage areas.\",\"imagePath\":\"hardware_hasps_&_latches_security_hasp_6_inch_20250620_201023.png\",\"name\":\"Security Hasp 6-inch\",\"price\":12.0,\"productType\":\"HASPS & LATCHES\",\"seasonalMultipliers\":\"1.0;1.0;1.1;1.3;1.2;1.1;1.0;1.0;1.4;1.8;1.6;1.2\",\"sku\":\"HWHL000036\"}}",
        "HWHL000036",
        0
      ],
      [
        "{\"labels\":[\"Store\"],\"oid\":\"13229323905400846\",\"properties\":{\"city\":\"Kirkland\",\"customerDistributionWeight\":4.0,\"orderFrequencyMultiplier\":1.4,\"orderValueMultiplier\":0.85,\"region\":\"Puget Sound\",\"rlsUserId\":\"9c8b7a65-4321-fed0-9876-543210fedcba\",\"state\":\"WA\",\"storeId\":\"STORE-KIR\",\"storeName\":\"Zava Retail Kirkland\",\"storeType\":\"Retail\"}}",
        "STORE-KIR",
        "{\"labels\":[\"Inventory\"],\"oid\":\"36310271995674632\",\"properties\":{\"availableQuantity\":0,\"inventoryId\":\"INV-STORE-KIR-HWHL000036\",\"lastRestockedAt\":\"2026-06-17T00:00:00.000+00:00\",\"quantityOnHand\":0,\"quantityReserved\":0,\"reorderPoint\":18,\"reorderQuantity\":96,\"sku\":\"HWHL000036\",\"storeId\":\"STORE-KIR\"}}",
        "INV-STORE-KIR-HWHL000036",
        "{\"labels\":[\"Product\"],\"oid\":\"1125899906842625\",\"properties\":{\"category\":\"HARDWARE\",\"description\":\"Heavy-duty hasp and staple set for padlock security on gates, sheds, and storage areas.\",\"imagePath\":\"hardware_hasps_&_latches_security_hasp_6_inch_20250620_201023.png\",\"name\":\"Security Hasp 6-inch\",\"price\":12.0,\"productType\":\"HASPS & LATCHES\",\"seasonalMultipliers\":\"1.0;1.0;1.1;1.3;1.2;1.1;1.0;1.0;1.4;1.8;1.6;1.2\",\"sku\":\"HWHL000036\"}}",
        "HWHL000036",
        0
      ],
      [
        "{\"labels\":[\"Store\"],\"oid\":\"13510798882111496\",\"properties\":{\"city\":\"Spokane\",\"customerDistributionWeight\":8.0,\"orderFrequencyMultiplier\":2.0,\"orderValueMultiplier\":1.0,\"region\":\"Eastern Washington\",\"rlsUserId\":\"d8e9f0a1-b2c3-4567-8901-234567890abc\",\"state\":\"WA\",\"storeId\":\"STORE-SPO\",\"storeName\":\"Zava Retail Spokane\",\"storeType\":\"Retail\"}}",
        "STORE-SPO",
        "{\"labels\":[\"Inventory\"],\"oid\":\"10414574138294277\",\"properties\":{\"availableQuantity\":0,\"inventoryId\":\"INV-STORE-SPO-GOPRN002\",\"lastRestockedAt\":\"2026-06-19T00:00:00.000+00:00\",\"quantityOnHand\":4,\"quantityReserved\":4,\"reorderPoint\":16,\"reorderQuantity\":36,\"sku\":\"GOPRN002\",\"storeId\":\"STORE-SPO\"}}",
        "INV-STORE-SPO-GOPRN002",
        "{\"labels\":[\"Product\"],\"oid\":\"1688849860263940\",\"properties\":{\"category\":\"GARDEN & OUTDOOR\",\"description\":\"Telescoping pole saw for reaching high branches safely from the ground.\",\"imagePath\":\"garden_&_outdoor_pruning_tools_pole_saw_pruner_8_foot_20250620_220454.png\",\"name\":\"Pole Saw Pruner 8-foot